# XGBoost 폭락/급등 전조 탐지기 — 빈칸 연습

**원본**: `notebooks/XGBoost_CrashSurge.ipynb`

`___` 빈칸을 채워서 코드를 완성하세요. 정답은 마지막 셀에 있습니다.

# XGBoost 폭락/급등 전조 신호 탐지기 — 3클래스 분류

## 목적
향후 **20영업일** 내 S&P 500의 **폭락(-10%)** 또는 **급등(+10%)** 발생 가능성을 사전에 감지하는 XGBoost 3클래스 분류 모델.

## 라벨 정의
| 라벨 | 의미 | 조건 |
|------|------|------|
| **0** | 정상 | 향후 20일 내 ±10% 이벤트 없음 |
| **1** | 폭락 전조 | 향후 20일 내 -10% 이하 하락 발생 |
| **2** | 급등 전조 | 향후 20일 내 +10% 이상 상승 발생 |

## 피처 (37개)
- **가격/수익률/추세** (8): 로그수익률 1/5/10/20일, 낙폭 60일, MA 괴리 50/200일, 일중 변동폭
- **실현변동성** (4): RV 5/21일, EWMA λ=0.94, 변동성의 변동성
- **옵션/내재변동성** (6): VIX(ln), VIX 변화, VIX 퍼센타일, VXV-VIX, SKEW, P/C비율
- **신용** (3): HY OAS, BBB OAS, CCC OAS
- **금리** (2): 10년 금리, 장단기 금리차
- **크로스에셋** (2): 달러지수 5일, WTI 5일
- **Tier 2 보조** (12): VIX9D-VIX, VVIX, P/C Equity, VRP, Parkinson, Amihud, 달러거래대금 z-score, 실질금리, BEI, SOFR-EFFR, NFCI, 주식-금리 상관

## 검증 기준
- 홀드아웃 macro F1 > 0.25
- 2008/2020 위기 전 crash_score > 90 (상위 10% percentile)
- 2020-21 반등기 surge_score > 80
- 안정기(2013~2019) 평균 < 55 (percentile 중앙값 근처)

## Q1. 라이브러리 import
- XGBoost 분류 모델 클래스명은?
- 피처 정규화 스케일러 클래스명은?
- 시계열 교차검증 클래스명은?
- F1 점수 함수명은?

In [ ]:
# ── Cell 1: 원시 데이터 수집 ──────────────────────────────────────────
!pip install imbalanced-learn optuna -q

import warnings, time
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import requests
import yfinance as yf
import xgboost as xgb
from xgboost import ___  # 빈칸: XGBoost 분류 모델 클래스
from sklearn.preprocessing import ___  # 빈칸: 표준화 스케일러 클래스
from sklearn.model_selection import ___  # 빈칸: 시계열 교차검증 클래스
from sklearn.metrics import (
    classification_report, confusion_matrix, ___,  # 빈칸: F1 점수 함수
    precision_recall_curve, average_precision_score,
)
import optuna
from imblearn.over_sampling import SMOTE

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
FRED_BASE = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id='



In [ ]:
# ── 헬퍼 ──
def fetch_fred(series_id: str, col_name: str, retries: int = 4, timeout: int = 30) -> pd.DataFrame:
    """FRED CSV 다운로드 (지수 백오프 재시도)."""
    url = FRED_BASE + series_id
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout)
            resp.raise_for_status()
            df = pd.read_csv(StringIO(resp.text), index_col=0, parse_dates=True)
            df.columns = [col_name]
            df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
            return df
        except Exception:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f'  [{series_id}] 재시도 {attempt+1}/{retries} ({wait}초 대기)...')
                time.sleep(wait)
            else:
                raise

def strip_tz(obj):
    """타임존 제거 (Series / DataFrame)."""
    obj = obj.copy()
    if hasattr(obj.index, 'tz') and obj.index.tz is not None:
        obj.index = obj.index.tz_localize(None)
    return obj

# ── 1) SPY OHLCV ──
print('▶ SPY OHLCV 수집...')
spy_raw = yf.Ticker('SPY').history(start='2000-01-01', auto_adjust=True)
spy = strip_tz(spy_raw)[['Open', 'High', 'Low', 'Close', 'Volume']]
print(f'  SPY: {spy.index[0].date()} ~ {spy.index[-1].date()} ({len(spy)}행)')

# ── 2) FRED 12개 시리즈 ──
FRED_MAP = {
    'BAMLH0A0HYM2': 'HY_OAS',
    'BAMLC0A4CBBB': 'BBB_OAS',
    'BAMLH0A3HYC':  'CCC_OAS',
    'DGS10':        'DGS10',
    'T10Y3M':       'T10Y3M',
    'DFII10':       'DFII10',
    'T10YIE':       'T10YIE',
    'SOFR':         'SOFR',
    'EFFR':         'EFFR',
    'NFCI':         'NFCI',
    'DTWEXBGS':     'DTWEXBGS',
    'DCOILWTICO':   'WTI',
}

fred = {}
print('\n▶ FRED 시리즈 수집...')
for sid, col in FRED_MAP.items():
    try:
        fred[col] = fetch_fred(sid, col)
        r = fred[col].dropna()
        print(f'  {col}: {r.index[0].date()} ~ {r.index[-1].date()} ({len(r)}행)')
    except Exception as e:
        print(f'  {col}: 실패 — {e}')
        fred[col] = pd.DataFrame({col: []}, index=pd.DatetimeIndex([]))

# ── 3) Cboe 5개 (yfinance) ──
CBOE_MAP = {'^VIX': 'VIX', '^VIX3M': 'VIX3M', '^VIX9D': 'VIX9D', '^VVIX': 'VVIX', '^SKEW': 'SKEW'}
cboe = {}
print('\n▶ Cboe 지수 수집...')
for ticker, col in CBOE_MAP.items():
    try:
        h = yf.Ticker(ticker).history(start='2000-01-01', auto_adjust=True)
        h = strip_tz(h)
        cboe[col] = h['Close'].rename(col)
        print(f'  {col}: {cboe[col].dropna().index[0].date()} ~ {cboe[col].dropna().index[-1].date()} ({cboe[col].dropna().shape[0]}행)')
    except Exception as e:
        print(f'  {col}: 실패 — {e}')
        cboe[col] = pd.Series(dtype=float, name=col)

# ── 4) Put/Call 비율 ──
print('\n▶ Put/Call 비율 수집...')
putcall = {}
for name, ticker in [('PUTCALL_TOTAL', '^PCALL'), ('PUTCALL_EQUITY', '^EPCALL')]:
    try:
        h = yf.Ticker(ticker).history(start='2000-01-01', auto_adjust=True)
        h = strip_tz(h)
        putcall[name] = h['Close'].rename(name)
        print(f'  {name}: OK ({putcall[name].dropna().shape[0]}행)')
    except Exception as e:
        print(f'  {name}: 실패 (보조 피처로 처리) — {e}')
        putcall[name] = pd.Series(dtype=float, name=name)

print('\n✅ 데이터 수집 완료')
print(f'   SPY: {len(spy)}행')
print(f'   FRED: {sum(len(v.dropna()) for v in fred.values())}행 (12시리즈)')
print(f'   Cboe: {sum(len(v.dropna()) for v in cboe.values())}행 (5지수)')
print(f'   Put/Call: {sum(len(v.dropna()) for v in putcall.values())}행')

## Q2. 가격/수익률/추세 피처 (8개)
- 로그수익률 = ?(close / close.shift(n))
- 60일 낙폭 = close / close.rolling(60).?() - 1
- MA 괴리율: 50일, 200일 이동평균
- 일중 변동폭 = (고가 - 저가) / 종가

In [ ]:
# ── Cell 2: 피처 엔지니어링 + 3클래스 라벨 생성 ──────────────────────

close = spy['Close']
high  = spy['High']
low   = spy['Low']
opn   = spy['Open']
vol   = spy['Volume']

feat = pd.DataFrame(index=spy.index)

# ━━━ A. 37개 피처 생성 ━━━

# ── 가격/수익률/추세 (8개) ──
for n in [1, 5, 10, 20]:
    feat[f'SP500_LOGRET_{n}D'] = ___(close / close.shift(n))  # 빈칸: 로그수익률 함수

feat['SP500_DRAWDOWN_60D'] = close / close.rolling(___).___() - 1  # 빈칸: 윈도우, 최고가 메소드
feat['SP500_MA_GAP_50']    = close / close.rolling(___).___() - 1  # 빈칸: 윈도우, 평균 메소드
feat['SP500_MA_GAP_200']   = close / close.rolling(___).___() - 1  # 빈칸: 윈도우, 평균 메소드
feat['SP500_INTRADAY_RANGE'] = (___ - ___) / close  # 빈칸: 고가, 저가 변수명



## Q3. 실현변동성 피처 (4개)
- 일별 수익률 = log(종가 / 전일종가)
- 연율화: std × √(?)
- EWMA λ=? (지수가중이동평균)
- 변동성의 변동성: RV_5D의 ?일 롤링 std

In [ ]:
# ── 실현변동성 (4개) ──
daily_ret = ___(close / close.shift(1))  # 빈칸: 로그수익률 함수
feat['RV_5D']  = daily_ret.rolling(___).___() * np.sqrt(___)  # 빈칸: 윈도우, 표준편차, 거래일
feat['RV_21D'] = daily_ret.rolling(___).___() * np.sqrt(___)  # 빈칸: 윈도우, 표준편차, 거래일

# EWMA λ=0.94
lam = ___  # 빈칸: EWMA 감쇠 계수 (0~1)
ewma_var = daily_ret.copy() * 0
ewma_var.iloc[0] = daily_ret.iloc[:21].___()  # 빈칸: 분산 메소드
for i in range(1, len(daily_ret)):
    ewma_var.iloc[i] = ___ * ewma_var.iloc[i-1] + (1 - ___) * daily_ret.iloc[i]**2  # 빈칸: λ, 1-λ
feat['EWMA_VOL_L94'] = np.___(ewma_var) * np.sqrt(___)  # 빈칸: 제곱근 함수, 거래일

feat['VOL_OF_VOL_21D'] = feat['RV_5D'].rolling(___).___()  # 빈칸: 윈도우, 표준편차



## Q4. 옵션/내재변동성 피처 (6개)
- VIX 레벨: ?(vix) (로그 변환)
- VIX 1일 변화율: .pct_change(?)
- VIX 252일 퍼센타일: .rolling(?).apply(rank)
- 결측치 처리: .?() (전방 채움)

In [ ]:
# ── 옵션/내재변동성 (6개) ──
vix_s = cboe.get('VIX', pd.Series(dtype=float))
feat['VIX_LEVEL']     = ___(vix_s.reindex(spy.index).___().clip(lower=1))  # 빈칸: 로그함수, 결측치 채움
feat['VIX_CHANGE_1D'] = vix_s.reindex(spy.index).ffill().pct_change(___)  # 빈칸: 변화율 기간
feat['VIX_PCTL_252D'] = vix_s.reindex(spy.index).ffill().rolling(___).apply(  # 빈칸: 퍼센타일 윈도우
    lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
)

vix3m_s = cboe.get('VIX3M', pd.Series(dtype=float))
feat['VXV_MINUS_VIX'] = vix3m_s.reindex(spy.index).ffill() - vix_s.reindex(spy.index).ffill()

skew_s = cboe.get('SKEW', pd.Series(dtype=float))
feat['SKEW_LEVEL'] = skew_s.reindex(spy.index).ffill()

pc_total = putcall.get('PUTCALL_TOTAL', pd.Series(dtype=float))
feat['PUTCALL_TOTAL'] = pc_total.reindex(spy.index).ffill()



## Q5. 신용/금리/크로스에셋 피처
- 크로스에셋 수익률: log(현재 / ?일전)
- 결측치 처리: .reindex().ffill()

In [ ]:
# ── 신용 (3개) ──
for col in ['HY_OAS', 'BBB_OAS', 'CCC_OAS']:
    feat[col] = fred[col][col].reindex(spy.index).ffill()

# ── 금리 (2개) ──
feat['DGS10_LEVEL']  = fred['DGS10']['DGS10'].reindex(spy.index).ffill()
feat['T10Y3M_SLOPE'] = fred['T10Y3M']['T10Y3M'].reindex(spy.index).ffill()

# ── 크로스에셋 (2개) ──
dxy_s = fred['DTWEXBGS']['DTWEXBGS'].reindex(spy.index).ffill()
feat['DTWEXBGS_RET_5D'] = ___(dxy_s / dxy_s.shift(___))  # 빈칸: 로그함수, 라그

wti_s = fred['WTI']['WTI'].reindex(spy.index).ffill()
feat['WTI_RET_5D'] = ___(wti_s / wti_s.shift(___))  # 빈칸: 로그함수, 라그



## Q6. Tier 2 보조 피처 (12개)
- Parkinson Vol: √( log(H/L)² / (4×log(2)) × ?일 롤링평균 × 252 )
- Amihud: |log(C/O)| / (C×V)
- Dollar Volume Z-score: (dv - mean) / std

In [ ]:
# ── Tier 2 보조 (12개) ──
vix9d_s = cboe.get('VIX9D', pd.Series(dtype=float))
feat['VIX9D_MINUS_VIX'] = vix9d_s.reindex(spy.index).ffill() - vix_s.reindex(spy.index).ffill()

vvix_s = cboe.get('VVIX', pd.Series(dtype=float))
feat['VVIX_LEVEL'] = vvix_s.reindex(spy.index).ffill()

pc_eq = putcall.get('PUTCALL_EQUITY', pd.Series(dtype=float))
feat['PUTCALL_EQUITY'] = pc_eq.reindex(spy.index).ffill()

# VRP: VIX²환산 - RV_21D²
vix_ann = (vix_s.reindex(spy.index).ffill() / ___)  # 빈칸: VIX %단위→소수 변환  # VIX는 %단위
feat['VARIANCE_RISK_PREMIUM'] = vix_ann**___ - (feat['RV_21D'] / np.sqrt(___))**___ * ___  # 빈칸: 제곱, 거래일, 제곱, 거래일

# Parkinson Vol 21D
feat['PARKINSON_VOL_21D'] = np.sqrt(
    (___(high / low)**2 / (4 * ___(2))).rolling(___).___() * ___  # 빈칸: 로그, 로그, 윈도우, 평균, 연율화
)

# Amihud Illiquidity 20D (SPY 단일 종목)
oc_ret = ___(close / opn).___()  # 빈칸: 로그함수, 절대값 메소드
dollar_vol = close * ___  # 빈칸: 거래량 변수명
amihud_daily = oc_ret / dollar_vol.replace(0, np.nan)
log_amihud = ___(amihud_daily.rolling(___).___() + 1e-15)  # 빈칸: 로그, 윈도우, 평균
feat['SP500_AMIHUD_ILLIQ_20D'] = log_amihud

# Dollar Volume Z-score 20D
dv = close * vol
dv_mean = dv.rolling(___).___()  # 빈칸: 윈도우, 평균
dv_std  = dv.rolling(___).___()  # 빈칸: 윈도우, 표준편차
feat['SP500_DOLLAR_VOLUME_Z_20D'] = (dv - dv_mean) / dv_std.replace(0, np.nan)

# FRED 보조
feat['DFII10_REAL10Y']   = fred['DFII10']['DFII10'].reindex(spy.index).ffill()
feat['T10YIE_BREAKEVEN'] = fred['T10YIE']['T10YIE'].reindex(spy.index).ffill()

sofr_s = fred['SOFR']['SOFR'].reindex(spy.index).ffill()
effr_s = fred['EFFR']['EFFR'].reindex(spy.index).ffill()
feat['SOFR_MINUS_EFFR'] = sofr_s - effr_s

nfci_s = fred['NFCI']['NFCI'].reindex(spy.index).ffill()  # 주간→ffill
feat['NFCI_LEVEL'] = nfci_s

# 주식수익률-금리변화 60일 상관
dgs10_chg = fred['DGS10']['DGS10'].reindex(spy.index).ffill().___()  # 빈칸: 차분 메소드
feat['CORR_EQ_DGS10_60D'] = daily_ret.rolling(___).___(dgs10_chg)  # 빈칸: 윈도우, 상관 메소드



## Q7. v2 신규 피처
- 크레딧 스프레드 변화: .diff(?) (5일, 20일)
- VIX 변화율: .pct_change(?)

In [ ]:
# ━━━ NEW: 추가 피처 (v2) ━━━

# ── 크레딧 스프레드 변화율 (6개) ──
for col in ['HY_OAS', 'BBB_OAS', 'CCC_OAS']:
    s = fred[col][col].reindex(spy.index).ffill()
    feat[f'{col}_CHG_5D'] = s.diff(___)  # 빈칸: 5일 변화
    feat[f'{col}_CHG_20D'] = s.diff(___)  # 빈칸: 20일 변화

# ── VIX 텀스트럭처 (3개) ──
vix_ff = vix_s.reindex(spy.index).ffill()
vix3m_ff = vix3m_s.reindex(spy.index).ffill()
vix9d_ff = vix9d_s.reindex(spy.index).ffill()

# VIX9D/VIX 비율 (단기 공포 vs 중기 공포)
feat['VIX9D_VIX_RATIO'] = vix9d_ff / vix_ff.clip(lower=1)
# VIX/VIX3M 비율 (컨탱고/백워데이션)
feat['VIX_VIX3M_RATIO'] = vix_ff / vix3m_ff.clip(lower=1)
# VIX 5일 변화율
feat['VIX_CHG_5D'] = vix_ff.pct_change(___)  # 빈칸: VIX 변화율 기간

print(f'피처 생성 완료: {feat.shape[1]}개')
print(feat.columns.tolist())



## Q8. 3클래스 라벨 생성
- 전방 관측 윈도우 = ?일
- 폭락 임계값 = ?%
- 급등 임계값 = ?%
- 미래 수익률→현재로: .shift(?)
- 향후 수익률 = .pct_change(?).shift(?)

In [ ]:
# ━━━ B. 3클래스 라벨 생성 ━━━
FORWARD_WINDOW = ___  # 빈칸: 전방 윈도우 (영업일)
CRASH_THRESHOLD = ___  # 빈칸: 폭락 임계값
SURGE_THRESHOLD = ___  # 빈칸: 급등 임계값

# 향후 20일 수익률
fwd_ret_20d = close.pct_change(FORWARD_WINDOW).shift(___)  # 빈칸: shift 방향 (미래→현재)

# 폭락/급등 시작일 식별
crash_dates = fwd_ret_20d[fwd_ret_20d <= ___].index  # 빈칸: 폭락 임계 변수
surge_dates = fwd_ret_20d[fwd_ret_20d >= ___].index  # 빈칸: 급등 임계 변수

# 기본값: 정상(0)
label = pd.Series(0, index=close.index, name='label')

# 급등 전조: 이벤트 시작일에서 역으로 20일 전 구간에 label=2 (먼저 설정)
for dt in surge_dates:
    loc = close.index.get_loc(dt)
    start = max(0, loc - FORWARD_WINDOW)
    label.iloc[start:loc + 1] = 2

# 폭락 전조: 이벤트 시작일에서 역으로 20일 전 구간에 label=1 (나중에 덮어씀 → 폭락 우선)
for dt in crash_dates:
    loc = close.index.get_loc(dt)
    start = max(0, loc - FORWARD_WINDOW)
    label.iloc[start:loc + 1] = 1

print(f'\n라벨 분포:')
for v, name in [(0, '정상'), (1, '폭락전조'), (2, '급등전조')]:
    cnt = (label == v).sum()
    print(f'  {name}({v}): {cnt} ({cnt/len(label)*100:.1f}%)')
print(f'  전체: {len(label)}')

In [ ]:
# ── Cell 3: 전처리 ─────────────────────────────────────────────────

# ── 핵심 피처 — 가격/변동성/FRED 기반 (긴 히스토리 확보) ──
CORE_FEATURES = [
    # 가격/수익률/추세 (8)
    'SP500_LOGRET_1D', 'SP500_LOGRET_5D', 'SP500_LOGRET_10D', 'SP500_LOGRET_20D',
    'SP500_DRAWDOWN_60D', 'SP500_MA_GAP_50', 'SP500_MA_GAP_200', 'SP500_INTRADAY_RANGE',
    # 실현변동성 (4)
    'RV_5D', 'RV_21D', 'EWMA_VOL_L94', 'VOL_OF_VOL_21D',
    # 신용 (3)
    'HY_OAS', 'BBB_OAS', 'CCC_OAS',
    # 금리 (2)
    'DGS10_LEVEL', 'T10Y3M_SLOPE',
]

# ── 보조 피처 — 늦은 시작, 불안정, 또는 외부 수집 실패 가능 ──
AUX_FEATURES = [
    'VIX_LEVEL', 'VIX_CHANGE_1D', 'VIX_PCTL_252D',
    'VXV_MINUS_VIX', 'SKEW_LEVEL', 'PUTCALL_TOTAL',
    'DTWEXBGS_RET_5D', 'WTI_RET_5D',
    'VIX9D_MINUS_VIX', 'VVIX_LEVEL', 'PUTCALL_EQUITY',
    'VARIANCE_RISK_PREMIUM', 'PARKINSON_VOL_21D',
    'SP500_AMIHUD_ILLIQ_20D', 'SP500_DOLLAR_VOLUME_Z_20D',
    'DFII10_REAL10Y', 'T10YIE_BREAKEVEN',
    'SOFR_MINUS_EFFR', 'NFCI_LEVEL', 'CORR_EQ_DGS10_60D',
    # v2 신규: 크레딧 스프레드 변화율 (6)
    'HY_OAS_CHG_5D', 'HY_OAS_CHG_20D',
    'BBB_OAS_CHG_5D', 'BBB_OAS_CHG_20D',
    'CCC_OAS_CHG_5D', 'CCC_OAS_CHG_20D',
    # v2 신규: VIX 텀스트럭처 (3)
    'VIX9D_VIX_RATIO', 'VIX_VIX3M_RATIO', 'VIX_CHG_5D',
]

ALL_FEATURES = CORE_FEATURES + AUX_FEATURES
print(f'핵심 피처: {len(CORE_FEATURES)}개, 보조 피처: {len(AUX_FEATURES)}개, 합계: {len(ALL_FEATURES)}개')



## Q9. 데이터 조립
- 핵심 피처 NaN: .dropna(subset=?)
- 보조 피처 NaN: .fillna(?)
- inf 대체: .replace([np.inf, -np.inf], ?)

In [ ]:
# ── 데이터 조립 ──
df_full = feat[ALL_FEATURES].copy()
df_full['label'] = label

before = len(df_full)
df_full = df_full.dropna(subset=___)  # 빈칸: 핵심 피처 리스트 변수
print(f'핵심 피처 NaN 제거: {before} → {len(df_full)} ({before - len(df_full)}행 제거)')

df_full[AUX_FEATURES] = df_full[AUX_FEATURES].fillna(___)  # 빈칸: NaN 대체값
df_full = df_full.replace([np.inf, -np.inf], ___).dropna(subset=ALL_FEATURES)  # 빈칸: inf 대체값
print(f'inf/NaN 정리 후: {len(df_full)}행 (최신: {df_full.index[-1].date()})')



## Q10. 데이터 분할 + 스케일러
- 시계열 CV: ?(n_splits=?)
- 스케일러: ?()

In [ ]:
# ── 학습용 df: 라벨 유효 구간 ──
df = df_full[df_full.index <= close.index[-FORWARD_WINDOW - 1]].copy()
print(f'라벨 유효 구간 (학습용): {len(df)}행 ({df.index[0].date()} ~ {df.index[-1].date()})')
print(f'추론용 df_full: {len(df_full)}행 ({df_full.index[0].date()} ~ {df_full.index[-1].date()})')

if len(df) == 0:
    raise ValueError('데이터가 비어있습니다.')

# ── 3분할: 학습 / 캘리브레이션 / 테스트 ──
HOLDOUT_DAYS = min(504, len(df) - 100)
CALIB_DAYS = min(1008, len(df) - HOLDOUT_DAYS - 100)  # 캘리브레이션용 ~4년 (Platt에 충분한 샘플)

test_df = df.iloc[-HOLDOUT_DAYS:]
calib_df = df.iloc[-(HOLDOUT_DAYS + CALIB_DAYS):-HOLDOUT_DAYS]
train_df = df.iloc[:-(HOLDOUT_DAYS + CALIB_DAYS)]

X_train, y_train = train_df[ALL_FEATURES].values, train_df['label'].values
X_calib, y_calib = calib_df[ALL_FEATURES].values, calib_df['label'].values
X_test,  y_test  = test_df[ALL_FEATURES].values, test_df['label'].values

# Optuna용 개발셋 = 학습 + 캘리브레이션 (모델 탐색용)
dev_df = df.iloc[:-HOLDOUT_DAYS]
X_dev, y_dev = dev_df[ALL_FEATURES].values, dev_df['label'].values

print(f'\n학습셋:       {len(train_df)} ({train_df.index[0].date()} ~ {train_df.index[-1].date()})')
print(f'캘리브레이션셋: {len(calib_df)} ({calib_df.index[0].date()} ~ {calib_df.index[-1].date()})')
print(f'테스트셋:      {len(test_df)} ({test_df.index[0].date()} ~ {test_df.index[-1].date()})')

tscv = ___(n_splits=___)  # 빈칸: CV 클래스, 분할 수
scaler = ___()  # 빈칸: 스케일러 클래스

print('\n── 클래스 분포 ──')
for name, y, n in [('학습셋', y_train, len(train_df)),
                    ('캘리브레이션', y_calib, len(calib_df)),
                    ('테스트셋', y_test, len(test_df))]:
    print(f'  {name}:')
    for v, lbl in [(0, '정상'), (1, '폭락전조'), (2, '급등전조')]:
        cnt = (y == v).sum()
        print(f'    {lbl}({v}): {cnt} ({cnt/n*100:.1f}%)')

## Q11. 가중치 + Platt 헬퍼
- 감쇠 계수 DECAY=?
- balanced weight: n / (?_classes × count)
- Platt에서 확률 클리핑: np.clip(p, ?, ?)
- 로지스틱 회귀 클래스명?

In [ ]:
# ── Cell 4: XGBoost 3클래스 학습 + Platt Scaling 캘리브레이션 ──────
from sklearn.linear_model import LogisticRegression
from scipy.special import logit as scipy_logit
from collections import Counter

# ── 지수 감쇠 가중치 생성 ──
DECAY = ___  # 빈칸: 감쇠 계수

def make_decay_weights(n):
    """최근 데이터에 높은 가중치: decay^(n-1-i)"""
    return ___ ** np.arange(n - 1, -1, -1)  # 빈칸: 감쇠 계수 변수

def calc_sample_weight_balanced(y):
    """sklearn balanced class weight -> sample weight"""
    counts = Counter(y)
    n = len(y)
    n_classes = len(counts)
    class_w = {c: n / (___ * cnt) for c, cnt in counts.items()}  # 빈칸: 분모 요소
    return np.array([class_w[yi] for yi in y])

# ── Platt Scaling 헬퍼 함수 ──
def fit_platt(raw_proba, y_true_binary):
    """Platt Scaling: raw probability -> calibrated probability via sigmoid."""
    raw_clipped = np.___(raw_proba, 1e-6, 1 - 1e-6)  # 빈칸: 클리핑 함수
    logits = scipy_logit(raw_clipped).reshape(-1, 1)  # 로짓 변환
    lr = ___(C=1e10, solver='lbfgs', max_iter=1000)  # 빈칸: 로지스틱회귀 클래스
    lr.___(logits, y_true_binary)  # 빈칸: 학습 메소드
    return lr

def apply_platt(lr, raw_proba):
    """Platt 보정 적용."""
    raw_clipped = np.clip(raw_proba, 1e-6, 1 - 1e-6)
    logits = scipy_logit(raw_clipped).reshape(-1, 1)
    return lr.predict_proba(logits)[:, 1]



## Q12. Optuna objective 함수
- XGBClassifier objective=?
- num_class=?
- eval_metric=?
- 스케일러 학습+변환, 변환만 메소드?
- F1 평균 방식?

In [ ]:
# ── Optuna objective (개발셋 = 학습+캘리브레이션) ──
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 7),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }

    fold_scores = []
    for train_idx, val_idx in tscv.split(X_dev):
        X_tr, X_vl = X_dev[train_idx], X_dev[val_idx]
        y_tr, y_vl = y_dev[train_idx], y_dev[val_idx]

        sc = ___()  # 빈칸: 스케일러 클래스
        X_tr_s = sc.___(X_tr)  # 빈칸: 학습+변환 메소드
        X_vl_s = sc.___(X_vl)  # 빈칸: 변환만 메소드

        # v2: class_weight + decay (no SMOTE)
        class_sw = calc_sample_weight_balanced(y_tr)
        decay_sw = make_decay_weights(len(y_tr))
        sw = class_sw * decay_sw

        model = XGBClassifier(
            objective='___', num_class=___, eval_metric='___',  # 빈칸: 목적함수, 클래스수, 평가지표
            random_state=42, verbosity=0, use_label_encoder=False, **params,
        )
        model.___(X_tr_s, y_tr, sample_weight=sw)  # 빈칸: 학습 메소드

        y_pred = model.___(X_vl_s)  # 빈칸: 예측 메소드
        fold_scores.append(f1_score(y_vl, y_pred, average='___'))  # 빈칸: F1 평균 방식

    return np.mean(fold_scores)



## Q13. Optuna 실행
- 최적화 방향? (F1 최대화)

In [ ]:
# ── Optuna 실행 ──
print('▶ Optuna 50 trials 시작...')
study = optuna.create_study(direction='___', study_name='crash_surge_xgb')  # 빈칸: 방향
study.optimize(objective, n_trials=50, show_progress_bar=True)

best = study.best_params
print(f'\n최적 macro F1: {study.best_value:.4f}')
print(f'최적 파라미터: {best}')



## Q14. 최종 모델 학습
- 스케일러 fit_transform?
- XGBClassifier objective, num_class, eval_metric?
- model.fit?

## Q15. Platt Scaling 캘리브레이션
- scaler.transform? model.predict_proba?

In [ ]:
# ── Platt Scaling 헬퍼 함수 ──
def fit_platt(raw_proba, y_true_binary):
    """Platt Scaling: raw probability -> calibrated probability via sigmoid."""
    raw_clipped = np.clip(raw_proba, 1e-6, 1 - 1e-6)
    logits = scipy_logit(raw_clipped).reshape(-1, 1)
    lr = LogisticRegression(C=1e10, solver='lbfgs', max_iter=1000)
    lr.fit(logits, y_true_binary)
    return lr

def apply_platt(lr, raw_proba):
    """Platt 보정 적용."""
    raw_clipped = np.clip(raw_proba, 1e-6, 1 - 1e-6)
    logits = scipy_logit(raw_clipped).reshape(-1, 1)
    return lr.predict_proba(logits)[:, 1]

# ── Optuna objective (개발셋 = 학습+캘리브레이션) ──
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 7),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }

    fold_scores = []
    for train_idx, val_idx in tscv.split(X_dev):
        X_tr, X_vl = X_dev[train_idx], X_dev[val_idx]
        y_tr, y_vl = y_dev[train_idx], y_dev[val_idx]

        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_vl_s = sc.transform(X_vl)

        # v2: class_weight + decay (no SMOTE)
        class_sw = calc_sample_weight_balanced(y_tr)
        decay_sw = make_decay_weights(len(y_tr))
        sw = class_sw * decay_sw

        model = XGBClassifier(
            objective='multi:softprob', num_class=3, eval_metric='mlogloss',
            random_state=42, verbosity=0, use_label_encoder=False, **params,
        )
        model.fit(X_tr_s, y_tr, sample_weight=sw)

        y_pred = model.predict(X_vl_s)
        fold_scores.append(f1_score(y_vl, y_pred, average='macro'))

    return np.mean(fold_scores)

# ── Optuna 실행 ──
print('▶ Optuna 50 trials 시작...')
study = optuna.create_study(direction='maximize', study_name='crash_surge_xgb')
study.optimize(objective, n_trials=50, show_progress_bar=True)

best = study.best_params
print(f'\n최적 macro F1: {study.best_value:.4f}')
print(f'최적 파라미터: {best}')

# ── 학습셋으로 최종 모델 학습 (캘리브레이션셋 제외) ──
print('\n▶ 학습셋으로 최종 모델 학습...')
scaler_final = StandardScaler()
X_train_s = scaler_final.fit_transform(X_train)

class_sw_final = calc_sample_weight_balanced(y_train)
decay_sw_final = make_decay_weights(len(y_train))
sw_final = class_sw_final * decay_sw_final

model_final = XGBClassifier(
    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
    random_state=42, verbosity=0, use_label_encoder=False, **best,
)
model_final.fit(X_train_s, y_train, sample_weight=sw_final)
print('✅ 최종 모델 학습 완료')

# ── Platt Scaling 캘리브레이션 ──
print('\n▶ Platt Scaling 캘리브레이션...')
X_calib_s = scaler_final.___(X_calib)  # 빈칸: 변환 메소드
calib_proba = model_final.___(X_calib_s)  # 빈칸: 확률 예측 메소드

# 폭락 Platt
platt_crash = fit_platt(calib_proba[:, 1], (y_calib == 1).astype(int))
# 급등 Platt
platt_surge = fit_platt(calib_proba[:, 2], (y_calib == 2).astype(int))

# 캘리브레이션 결과 확인
crash_cal = apply_platt(platt_crash, calib_proba[:, 1])
surge_cal = apply_platt(platt_surge, calib_proba[:, 2])

print(f'  캘리브레이션셋 {len(calib_df)}일 사용')
print(f'  폭락 — 원래 확률 평균: {calib_proba[:, 1].mean()*100:.1f}% → 보정 후: {crash_cal.mean()*100:.1f}%')
print(f'  급등 — 원래 확률 평균: {calib_proba[:, 2].mean()*100:.1f}% → 보정 후: {surge_cal.mean()*100:.1f}%')

# Platt 파라미터 출력
print(f'\n  Platt 파라미터:')
print(f'    폭락: w={platt_crash.coef_[0][0]:.4f}, b={platt_crash.intercept_[0]:.4f}')
print(f'    급등: w={platt_surge.coef_[0][0]:.4f}, b={platt_surge.intercept_[0]:.4f}')
print('✅ Platt Scaling 캘리브레이션 완료')


## Q16. 홀드아웃 테스트
- scaler.?() model.?() model.?()
- F1 평균 방식?

In [ ]:
# ── Cell 5: 검증 & 5패널 시각화 (Crash: raw→Rank, Surge: Platt→Rank) ─────

# ── 홀드아웃 테스트셋 평가 (원래 모델) ──
X_test_s = scaler_final.___(X_test)  # 빈칸: 변환
y_proba_test_raw = model_final.___(X_test_s)  # 빈칸: 확률 예측
y_pred_test = model_final.___(X_test_s)  # 빈칸: 클래스 예측

CLASS_NAMES = ['정상(0)', '폭락전조(1)', '급등전조(2)']
print('=' * 60)
print('홀드아웃 테스트셋 Classification Report (원래 모델)')
print('=' * 60)
print(classification_report(y_test, y_pred_test, target_names=CLASS_NAMES, digits=3))

macro_f1 = f1_score(y_test, y_pred_test, average='___')  # 빈칸: 평균 방식
print(f'Macro F1-score: {macro_f1:.4f}  (기준: > 0.25)')



## Q17. 전체 기간 추론 + Percentile Rank
- scaler.transform? model.predict_proba?
- Percentile Rank: .rank(pct=?) × ?

In [ ]:
# ── 전체 기간 추론 ──
X_all_s = scaler_final.___(df_full[ALL_FEATURES].values)  # 빈칸: 변환
proba_all_raw = model_final.___(X_all_s)  # 빈칸: 확률 예측

# 원래 확률 (%)
crash_all_raw = pd.Series(proba_all_raw[:, 1] * 100, index=df_full.index)
surge_all_raw = pd.Series(proba_all_raw[:, 2] * 100, index=df_full.index)

# ── Platt Scaling: Surge만 적용 (Crash는 Platt weight 음수 → 스킵) ──
surge_platt = pd.Series(apply_platt(platt_surge, proba_all_raw[:, 2]) * 100, index=df_full.index)

# Platt 파라미터 확인
platt_crash_w = platt_crash.coef_[0][0]
print(f'\n  Platt crash w={platt_crash_w:.4f} ({"양수 ✅" if platt_crash_w > 0 else "음수 → Platt 스킵, raw 사용 ⚠️"})')
print(f'  Platt surge w={platt_surge.coef_[0][0]:.4f}')

# ── Percentile Rank 변환 (0~100) ──
# Crash: raw probability → Percentile Rank (Platt 없이)
# Surge: Platt 보정 → Percentile Rank
crash_all = crash_all_raw.___(pct=___) * ___  # 빈칸: 순위 메소드, 퍼센트, 스케일
surge_all = surge_platt.___(pct=___) * ___  # 빈칸: 순위 메소드, 퍼센트, 스케일

print(f'\n추론 범위: {df_full.index[0].date()} ~ {df_full.index[-1].date()} ({len(df_full)}행)')
print(f'\n── 변환 파이프라인 ──')
print(f'  폭락: raw 평균 {crash_all_raw.mean():.1f}% → (Platt 스킵) → rank {crash_all.mean():.1f}')
print(f'  급등: raw 평균 {surge_all_raw.mean():.1f}% → Platt {surge_platt.mean():.1f}% → rank {surge_all.mean():.1f}')



In [ ]:
# ── 5패널 차트 ──
fig, axes = plt.subplots(3, 2, figsize=(18, 20))
fig.suptitle('XGBoost 폭락/급등 전조 신호 — Crash: raw→Rank, Surge: Platt→Rank', fontsize=15, fontweight='bold')

# Panel 1: Confusion Matrix
ax = axes[0, 0]
cm = confusion_matrix(y_test, y_pred_test)
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('Confusion Matrix (3클래스)', fontsize=13)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
ax.set_xticklabels(['정상', '폭락', '급등'])
ax.set_yticklabels(['정상', '폭락', '급등'])
for i in range(3):
    for j in range(3):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=14)
fig.colorbar(im, ax=ax)

# Panel 2: 폭락 P-R Curve (raw probability 사용)
ax = axes[0, 1]
crash_true = (y_test == 1).astype(int)
prec_c, rec_c, _ = precision_recall_curve(crash_true, y_proba_test_raw[:, 1])
ap_c = average_precision_score(crash_true, y_proba_test_raw[:, 1])
ax.plot(rec_c, prec_c, 'r-', linewidth=2)
ax.fill_between(rec_c, prec_c, alpha=0.2, color='red')
ax.set_title(f'폭락 신호 P-R Curve (AP={ap_c:.3f}, raw)', fontsize=13)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1]); ax.grid(True, alpha=0.3)

# Panel 3: 급등 P-R Curve (Platt 보정)
ax = axes[1, 0]
surge_true = (y_test == 2).astype(int)
surge_prob_cal = apply_platt(platt_surge, y_proba_test_raw[:, 2])
prec_s, rec_s, _ = precision_recall_curve(surge_true, surge_prob_cal)
ap_s = average_precision_score(surge_true, surge_prob_cal)
ax.plot(rec_s, prec_s, 'g-', linewidth=2)
ax.fill_between(rec_s, prec_s, alpha=0.2, color='green')
ax.set_title(f'급등 신호 P-R Curve (AP={ap_s:.3f}, Platt)', fontsize=13)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1]); ax.grid(True, alpha=0.3)

# Panel 4: SPY 가격 + Percentile Rank
ax1 = axes[1, 1]
test_dates = test_df.index
ax1.plot(test_dates, close.reindex(test_dates), 'k-', linewidth=1, alpha=0.8, label='SPY')
ax1.set_title('SPY 가격 + Percentile Rank 신호', fontsize=13)
ax1.set_ylabel('SPY 가격', color='black')
ax2 = ax1.twinx()
ax2.fill_between(test_dates, crash_all.reindex(test_dates), alpha=0.4, color='red', label='폭락 순위점수')
ax2.fill_between(test_dates, surge_all.reindex(test_dates), alpha=0.4, color='green', label='급등 순위점수')
ax2.set_ylabel('Percentile Rank (0~100)', color='gray'); ax2.set_ylim([0, 100])
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

# Panel 5: 피처 중요도 Top 15
ax = axes[2, 0]
importances = model_final.feature_importances_
feat_imp = pd.Series(importances, index=ALL_FEATURES).sort_values(ascending=True)
top15 = feat_imp.tail(15)
colors = ['#d62728' if f in CORE_FEATURES else '#2ca02c' for f in top15.index]
top15.plot(kind='barh', ax=ax, color=colors)
ax.set_title('피처 중요도 Top 15 (빨강=핵심, 초록=보조)', fontsize=13)
ax.set_xlabel('Importance')

# Panel 6: 전체 기간 Percentile Rank + 주요 이벤트
ax = axes[2, 1]
ax.plot(df_full.index, crash_all, 'r-', alpha=0.6, linewidth=0.8, label='폭락 순위점수')
ax.plot(df_full.index, surge_all, 'g-', alpha=0.6, linewidth=0.8, label='급등 순위점수')
ax.axhline(90, color='red', linestyle='--', alpha=0.4, label='경고 (90)')
ax.axhline(80, color='orange', linestyle='--', alpha=0.3, label='주의 (80)')
ax.axhline(50, color='gray', linestyle='--', alpha=0.2)
events = {'2008-09-15': '리먼 파산', '2020-02-20': '코로나 폭락', '2020-03-23': '코로나 저점'}
for date_str, label_text in events.items():
    dt = pd.Timestamp(date_str)
    if dt in df_full.index:
        ax.axvline(dt, color='blue', alpha=0.4, linestyle=':')
        ax.annotate(label_text, xy=(dt, 95), fontsize=8, color='blue', rotation=90, va='top', ha='right')
ax.set_title('전체 기간 Percentile Rank + 주요 이벤트', fontsize=13)
ax.set_ylabel('Percentile Rank (0~100)'); ax.legend(loc='upper left', fontsize=8); ax.set_ylim([0, 100])

plt.tight_layout()
plt.show()

# ── 주요 이벤트 시점 (Percentile Rank) ──
print('\n── 주요 이벤트 시점 신호 (Percentile Rank) ──')
check_dates = ['2008-09-01', '2008-09-15', '2020-02-14', '2020-02-20',
               '2020-03-23', '2020-04-15', '2021-01-04']
for d in check_dates:
    dt = pd.Timestamp(d)
    idx = df_full.index.get_indexer([dt], method='nearest')[0]
    actual_dt = df_full.index[idx]
    cs = crash_all.loc[actual_dt]
    ss = surge_all.loc[actual_dt]
    cs_raw = crash_all_raw.loc[actual_dt]
    ss_p = surge_platt.loc[actual_dt]
    print(f'  {actual_dt.date()}: crash={cs:.1f} (raw {cs_raw:.1f}%), surge={ss:.1f} (Platt {ss_p:.1f}%)')

## Q18. 신호 등급 분류
- 등급 경계: ?=낮음, ?=보통, ?=주의, ?=경고, 이상=위험

In [ ]:
# ── Cell 6: 신호 강도 & 등급 (Percentile Rank 기반) ────────────

# ── 점수: Platt 보정 확률의 Percentile Rank (0~100) ──
crash_score = crash_all
surge_score = surge_all

# ── 등급 매핑 (Percentile 기반) ──
def grade(score):
    """0~100 percentile rank -> 등급"""
    if score < ___:  # 빈칸: 낮음 경계
        return '낮음'
    elif score < ___:  # 빈칸: 보통 경계
        return '보통'
    elif score < ___:  # 빈칸: 주의 경계
        return '주의'
    elif score < ___:  # 빈칸: 경고 경계
        return '경고'
    else:
        return '위험'



In [ ]:
crash_grade = crash_score.apply(grade)
surge_grade = surge_score.apply(grade)

# ── 최근 60일 듀얼 신호 타임라인 ──
last60 = df_full.index[-60:]

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(last60, crash_score.reindex(last60), alpha=0.5, color='red', label='폭락 신호')
ax.fill_between(last60, surge_score.reindex(last60), alpha=0.5, color='green', label='급등 신호')
ax.axhline(90, color='red', linestyle='--', alpha=0.5, linewidth=0.8)
ax.axhline(80, color='orange', linestyle='--', alpha=0.4, linewidth=0.8)
ax.axhline(50, color='gray', linestyle='--', alpha=0.3, linewidth=0.8)
ax.set_title('최근 60일 폭락/급등 신호 강도 (Percentile Rank)', fontsize=14, fontweight='bold')
ax.set_ylabel('순위 점수 (0~100)')
ax.set_ylim([0, 100])
ax.legend(loc='upper left')
ax.grid(True, alpha=0.2)

for y, txt in [(25, '낮음'), (60, '보통'), (77, '주의'), (90, '경고'), (97, '위험')]:
    ax.text(last60[-1], y, f'  {txt}', fontsize=8, color='gray', va='center')

plt.tight_layout()
plt.show()

# ── 최근 10일 신호 테이블 ──
last10 = df_full.index[-10:]
table = pd.DataFrame({
    '날짜': last10.date,
    'crash_score': crash_score.reindex(last10).round(1).values,
    'crash_grade': crash_grade.reindex(last10).values,
    'surge_score': surge_score.reindex(last10).round(1).values,
    'surge_grade': surge_grade.reindex(last10).values,
})
print('── 최근 10일 신호 테이블 ──')
print(table.to_string(index=False))

# ── 최신 점수 + 등급 출력 ──
latest = df_full.index[-1]
cs_latest = crash_score.loc[latest]
ss_latest = surge_score.loc[latest]
print(f'\n{"=" * 50}')
print(f'최신 날짜: {latest.date()}')
print(f'  폭락 신호: {cs_latest:.1f}점 ({grade(cs_latest)}) — 과거 대비 상위 {100-cs_latest:.0f}%')
print(f'  급등 신호: {ss_latest:.1f}점 ({grade(ss_latest)}) — 과거 대비 상위 {100-ss_latest:.0f}%')
print(f'{"=" * 50}')

# ── 안정기 평균 확인 ──
stable_mask = (df_full.index >= '2013-01-01') & (df_full.index <= '2019-12-31')
if stable_mask.any():
    stable_crash = crash_score[stable_mask].mean()
    stable_surge = surge_score[stable_mask].mean()
    print(f'\n안정기(2013~2019) 평균:')
    print(f'  crash_score: {stable_crash:.1f}  (기준: < 55)')
    print(f'  surge_score: {stable_surge:.1f}  (기준: < 55)')

# ── 위기 시점 검증 ──
print(f'\n── 위기 시점 검증 ──')
crisis_checks = [
    ('2008-09-15', 'crash', 90, '리먼 파산'),
    ('2020-02-20', 'crash', 90, '코로나 폭락 직전'),
    ('2020-04-15', 'surge', 80, '코로나 반등기'),
]
for d, signal, threshold, desc in crisis_checks:
    dt = pd.Timestamp(d)
    idx = df_full.index.get_indexer([dt], method='nearest')[0]
    actual_dt = df_full.index[idx]
    val = crash_score.loc[actual_dt] if signal == 'crash' else surge_score.loc[actual_dt]
    status = '✅' if val >= threshold else '❌'
    print(f'  {status} {desc} ({actual_dt.date()}): {signal}_score={val:.1f} (기준: > {threshold})')


---
## 정답

| 문제 | 정답 |
|------|------|
| Q1 | `XGBClassifier` |
| Q1 | `StandardScaler` |
| Q1 | `TimeSeriesSplit` |
| Q1 | `f1_score` |
| Q2 | `np.log` |
| Q2 | `max` |
| Q2 | `50` |
| Q2 | `200` |
| Q3 | `np.log` |
| Q3 | `252` |
| Q3 | `5` |
| Q3 | `21` |
| Q3 | `0.94` |
| Q4 | `np.log` |
| Q4 | `1` |
| Q4 | `252` |
| Q4 | `ffill` |
| Q5 | `np.log` |
| Q5 | `5` |
| Q6 | `np.log` |
| Q6 | `21` |
| Q6 | `20` |
| Q6 | `np.sqrt` |
| Q6 | `252` |
| Q7 | `5` |
| Q7 | `20` |
| Q7 | `5` |
| Q8 | `20` |
| Q8 | `-0.10` |
| Q8 | `0.10` |
| Q8 | `-FORWARD_WINDOW` |
| Q9 | `CORE_FEATURES` |
| Q9 | `0` |
| Q9 | `np.nan` |
| Q10 | `TimeSeriesSplit` |
| Q10 | `5` |
| Q10 | `StandardScaler` |
| Q11 | `0.9995` |
| Q11 | `n_classes` |
| Q11 | `1e-6` |
| Q11 | `1 - 1e-6` |
| Q11 | `LogisticRegression` |
| Q12 | `multi:softprob` |
| Q12 | `3` |
| Q12 | `mlogloss` |
| Q12 | `fit_transform` |
| Q12 | `transform` |
| Q12 | `macro` |
| Q13 | `maximize` |
| Q14 | `fit_transform` |
| Q14 | `multi:softprob` |
| Q14 | `3` |
| Q14 | `mlogloss` |
| Q14 | `.fit()` |
| Q15 | `transform` |
| Q15 | `predict_proba` |
| Q16 | `transform` |
| Q16 | `predict_proba` |
| Q16 | `predict` |
| Q16 | `macro` |
| Q17 | `transform` |
| Q17 | `predict_proba` |
| Q17 | `True` |
| Q17 | `100` |
| Q18 | `50` |
| Q18 | `70` |
| Q18 | `85` |
| Q18 | `95` |